# Ciclo de Vida de ML (CRISP-ML) - ETL (Extracción, Transformación y Carga)

Este cuaderno detalla la fase de **Preparación de Datos** dentro del ciclo de vida del aprendizaje automático. En esta fase, cargamos el conjunto de datos crudo de calificaciones y realizamos las transformaciones necesarias para limpiarlo y estructurarlo adecuadamente.

### Paso 1: Importación de librerías
Cargamos las librerías esenciales para manipulación de datos (`pandas`, `numpy`) y gestión de archivos.

In [ ]:
# Importación de librerías del sistema
import pandas as pd
import numpy as np
import os

print("Librerías cargadas exitosamente.")

### Paso 2: Análisis Inicial del Dataset Crudo
Inspeccionamos el delimitador del archivo de origen y sus columnas. El dataset original contiene registros de calificaciones escolares desde 2015 hasta 2024. Leemos las primeras 5 filas para examinar el formato general.

In [ ]:
raw_csv_path = 'Resultados2015-2024.csv'

# Visualizar la cabecera y el delimitador original (;)
df_preview = pd.read_csv(raw_csv_path, sep=';', nrows=5)
print("Columnas detectadas:", df_preview.columns.tolist())
df_preview.head()

### Paso 3: Limpieza y Normalización por Chunks
Dado que el archivo tiene un tamaño de ~85MB y contiene más de 840,000 registros, procesaremos los datos en bloques (chunks) de 150,000 filas para garantizar eficiencia de memoria. En cada bloque realizamos las siguientes transformaciones:
1. Coerción de columnas numéricas clave como `nota`, `faltas` y `refuerzo` (rellenando vacíos con 0 en faltas y refuerzos).
2. Conversión a tipos de datos de bajo consumo físico (`int` y `float`).
3. Homogeneización de la columna objetivo `promovido` filtrando registros incompletos o erróneos.

In [ ]:
cols_to_use = [
    'IdFila', 'ano', 'matricula', 'grado', 'promovido', 
    'codmat', 'nommat', 'intensidad_semannal', 'faltas', 
    'nota', 'periodo', 'refuerzo', 'ultimo_periodo'
]

chunks = []
chunk_idx = 0

# Procesamiento iterativo eficiente en memoria
for chunk in pd.read_csv(raw_csv_path, sep=';', usecols=cols_to_use, low_memory=False, chunksize=150000):
    chunk_idx += 1
    # Descartar filas donde las llaves principales sean nulas
    chunk = chunk.dropna(subset=['IdFila', 'ano', 'matricula'])
    
    # Convertir identificadores y limpiar tipos de datos numéricos
    chunk['IdFila'] = pd.to_numeric(chunk['IdFila'], errors='coerce')
    chunk['ano'] = pd.to_numeric(chunk['ano'], errors='coerce')
    chunk['matricula'] = pd.to_numeric(chunk['matricula'], errors='coerce')
    chunk = chunk.dropna(subset=['IdFila', 'ano', 'matricula'])
    
    chunk['IdFila'] = chunk['IdFila'].astype(int)
    chunk['ano'] = chunk['ano'].astype(int)
    chunk['matricula'] = chunk['matricula'].astype(int)
    
    # Calificaciones, inasistencias y clases de recuperación
    chunk['nota'] = pd.to_numeric(chunk['nota'], errors='coerce')
    chunk['faltas'] = pd.to_numeric(chunk['faltas'], errors='coerce').fillna(0).astype(int)
    chunk['refuerzo'] = pd.to_numeric(chunk['refuerzo'], errors='coerce').fillna(0).astype(int)
    chunk['intensidad_semannal'] = pd.to_numeric(chunk['intensidad_semannal'], errors='coerce').fillna(1).astype(int)
    chunk['periodo'] = pd.to_numeric(chunk['periodo'], errors='coerce').fillna(1).astype(int)
    chunk['ultimo_periodo'] = pd.to_numeric(chunk['ultimo_periodo'], errors='coerce').fillna(5).astype(int)
    
    # Normalización del grado del estudiante
    chunk['grado'] = chunk['grado'].astype(str).str.strip()
    
    # Estandarización de la variable promovido a ('S', 'N')
    chunk['promovido'] = chunk['promovido'].astype(str).str.strip().str.upper()
    chunk.loc[~chunk['promovido'].isin(['S', 'N']), 'promovido'] = np.nan
    
    # Normalización del nombre de la asignatura
    chunk['nommat'] = chunk['nommat'].astype(str).str.strip().str.upper()
    chunks.append(chunk)

# Concatenar todos los chunks en un solo DataFrame
df = pd.concat(chunks, ignore_index=True)
print(f"Carga inicial lista. Registros cargados: {len(df)}")

### Paso 4: Limpieza e Imputación de Asignaturas (`nommat`)
Muchos nombres de asignaturas contienen errores de codificación (caracteres extraños o corruptos). Aplicamos un diccionario de mapeo para corregirlos, eliminamos códigos puramente numéricos de materias y completamos notas faltantes con la mediana de la materia correspondiente.

In [ ]:
subject_mapping = {
    'CIENCIASNATURALESYEDUCACINAMBIENTAL': 'CIENCIAS NATURALES Y EDUCACIÓN AMBIENTAL',
    'CIENCIASPOLITICASYECONOMICAS': 'CIENCIAS POLÍTICAS Y ECONÓMICAS',
    'CIENCIASSOCIALES(HISTORIA-GEOGRAFA-CONSTITUCINPOLTICAYDEMOCRACIA.)': 'CIENCIAS SOCIALES (HISTORIA, GEOGRAFÍA, CONSTITUCIÓN POLÍTICA Y DEMOCRACIA)',
    'DISEOEINTEGRACIONDEMULTIMEDIA': 'DISEÑO E INTEGRACIÓN DE MULTIMEDIA',
    'EDUCACIONETICAYENVALORESHUMANOS': 'EDUCACIÓN ÉTICA Y VALORES HUMANOS',
    'EDUCACINARTISTICAYCULTURAL': 'EDUCACIÓN ARTÍSTICA Y CULTURAL',
    'EDUCACINFSICA-RECREACINYDEPORTES': 'EDUCACIÓN FÍSICA, RECREACIÓN Y DEPORTES',
    'HUMANIDADESIDIOMAEXTRANJERO(INGLS)': 'HUMANIDADES IDIOMA EXTRANJERO (INGLÉS)',
    'IDIOMAEXTRANJERO(INGLS)': 'IDIOMA EXTRANJERO (INGLÉS)',
    'INVESTIGACIN': 'INVESTIGACIÓN',
    'LOGSTICA': 'LOGÍSTICA',
    'MATEMTICA': 'MATEMÁTICAS',
    'MATEMTICAS': 'MATEMÁTICAS',
    'QUMICA': 'QUÍMICA',
    'FSICA': 'FÍSICA',
    'TECNOLOGIAEINFORMTICA': 'TECNOLOGÍA E INFORMÁTICA',
    'TCNICOENASISTENCIAADMINISTRATIVA': 'TÉCNICO EN ASISTENCIA ADMINISTRATIVA',
    'TCNICOENRECURSOSHUMANOS': 'TÉCNICO EN RECURSOS HUMANOS',
    'TCNICOPATRONISTAINDUSTRIALENCONFECCIN': 'TÉCNICO PATRONISTA INDUSTRIAL EN CONFECCIÓN',
    'CATEDRADEPAZ"SOSTENIBILIDADAMBIENTAL"': 'CÁTEDRA DE PAZ Y SOSTENIBILIDAD AMBIENTAL',
    'PROYECTODEVIDA(EREYEV)': 'PROYECTO DE VIDA'
}

df['nommat'] = df['nommat'].replace(subject_mapping)

# Conservar solo asignaturas válidas con texto (mínimo 3 letras)
df = df[df['nommat'].str.contains('[A-Z]{3,}', na=False, regex=True)]

# Imputación de notas nulas con la mediana específica de cada asignatura
sub_medians = df.groupby('nommat')['nota'].median()
overall_median = df['nota'].median()
df['nota'] = df['nota'].fillna(df['nommat'].map(sub_medians)).fillna(overall_median)

# Crear la bandera de materia perdida (calificación menor a 3.0)
df['es_perdida'] = (df['nota'] < 3.0).astype(int)

# Eliminar registros sin variable de promoción (necesaria para el modelado clasificador)
df_clean = df.dropna(subset=['promovido']).copy()
print(f"Filas tras imputación y limpieza: {len(df_clean)}")

### Paso 5: Carga (Guardar Dataset Transaccional Limpio)
Guardamos el conjunto de datos limpio transaccional en un archivo CSV llamado `resultados_limpios.csv` usando codificación UTF-8.

In [ ]:
clean_csv_path = 'resultados_limpios.csv'
df_clean.to_csv(clean_csv_path, sep=';', index=False, encoding='utf-8')
print(f"Dataset limpio guardado exitosamente en: {clean_csv_path}")

### Paso 6: Agregación a nivel Estudiante-Año
Para un modelado predictivo más eficiente, agrupamos las calificaciones transaccionales por estudiante (`matricula`) y por año lectivo (`ano`). Calculamos características agregadas como:
- Calificación promedio final (`promedio_nota`)
- Calificación mínima y máxima obtenida (`nota_min`, `nota_max`)
- Total de faltas acumuladas (`total_faltas`)
- Sesiones de refuerzo tomadas (`total_refuerzos`)
- Cantidad de asignaturas reprobadas (`materias_reprobadas`)
- Número de materias totales cursadas (`total_materias`)
- Estado de promoción final (`promovido` y `promovido_num` como 1/0).

In [ ]:
print("Agrupando datos por estudiante y año escolar...")

student_year = df_clean.groupby(['ano', 'matricula']).agg(
    grado=('grado', 'first'),
    promovido=('promovido', 'first'),
    promedio_nota=('nota', 'mean'),
    nota_min=('nota', 'min'),
    nota_max=('nota', 'max'),
    total_faltas=('faltas', 'sum'),
    total_refuerzos=('refuerzo', 'sum'),
    materias_reprobadas=('es_perdida', 'sum'),
    total_materias=('nommat', 'nunique')
).reset_index()

# Conversión numérica del objetivo para clasificación
student_year['promovido_num'] = student_year['promovido'].map({'S': 1, 'N': 0})

agg_csv_path = 'resultados_estudiantes.csv'
student_year.to_csv(agg_csv_path, sep=';', index=False, encoding='utf-8')
print(f"Dataset agregado guardado. Filas: {len(student_year)} en {agg_csv_path}")